# Stability — feature-importance distance, parameter distance

Bootstraps each model to measure how much its feature importances, parameters, and
performance shift under resampled training data, for whichever models have a
`models/<name>_model.py` file so far.

**Compute note**: bootstrapping refits a model `N_BOOT` times. If a single fit is
slow (especially for a heavily-tuned XGBoost or a fine-tuning TabPFN), lower
`N_BOOT` or subsample `X_train` before bootstrapping — this is a scope call, not a
correctness one.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from common_metrics import (
    FEATURES, MODEL_NAMES, TEAM_THRESHOLD,
    load_split, get_X_y, available_models, report_status, importance_vector,
)

report_status()

## Load data

In [ ]:
train_df = load_split("train")
test_df = load_split("test")

X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## Feature-importance distance — the shared metric across all 3 models

`d(f1, f2) = ||φ(f1) - φ(f2)||₂` — model-agnostic via `common_metrics.importance_vector`,
so this is the one number comparable across xgboost/logreg/tabpfn.

**TabPFN nuance**: if TabPFN's own `fit()` re-subsamples its ~10K-row context on
every call, bootstrapping here (which resamples `X_train` before calling `fit()`)
tests sensitivity to *which* subsample gets drawn — that's the more informative
question for this dataset's scale. If instead `fit()` uses a fixed, pre-chosen
context, this only tests ordinary sampling noise. Confirm which applies once the
TabPFN owner's module lands.

In [ ]:
def stability_bootstrap(module, model_type, X_train, y_train, n_boot=30, random_state=42):
    rng = np.random.RandomState(random_state)
    base_model = module.fit(X_train, y_train)
    base_imp = importance_vector(model_type, base_model, module, X_train, y_train)

    distances = []
    for i in range(n_boot):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]
        model_b = module.fit(X_b, y_b)
        imp_b = importance_vector(model_type, model_b, module, X_b, y_b)
        distances.append(float(np.linalg.norm(base_imp - imp_b)))
    return distances

## Parameter distance — logreg only

In [ ]:
def parameter_distance_bootstrap(module, X_train, y_train, n_boot=30, random_state=42):
    rng = np.random.RandomState(random_state)
    base_model = module.fit(X_train, y_train)
    theta_base = np.asarray(base_model.coef_[0])
    distances = []
    for i in range(n_boot):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]
        theta_b = np.asarray(module.fit(X_b, y_b).coef_[0])
        distances.append(float(np.linalg.norm(theta_base - theta_b)))
    return distances

## Performance stability — AUC spread across the same bootstraps

In [ ]:
def performance_stability_bootstrap(module, X_train, y_train, X_test, y_test, n_boot=30, random_state=42):
    rng = np.random.RandomState(random_state)
    aucs = []
    for i in range(n_boot):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]
        model_b = module.fit(X_b, y_b)
        probs = module.predict_proba(model_b, X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, probs))
    return aucs

## Run across available models

In [ ]:
MODEL_TYPE = {"xgboost": "xgboost", "logreg": "logreg", "tabpfn": "tabpfn"}
N_BOOT = 30  # lower this (or subsample X_train) if a single fit is slow

stability_results = {}
for name, module in available_models().items():
    print(f"\n=== {name} ===")
    model_type = MODEL_TYPE[name]
    imp_distances = stability_bootstrap(module, model_type, X_train, y_train, n_boot=N_BOOT)
    auc_spread = performance_stability_bootstrap(module, X_train, y_train, X_test, y_test, n_boot=N_BOOT)
    stability_results[name] = {"importance_distance": imp_distances, "auc_spread": auc_spread}
    print(f"  importance distance: mean={np.mean(imp_distances):.4f}, std={np.std(imp_distances):.4f}")
    print(f"  AUC spread: mean={np.mean(auc_spread):.4f}, std={np.std(auc_spread):.4f}")

    if name == "logreg":
        param_distances = parameter_distance_bootstrap(module, X_train, y_train, n_boot=N_BOOT)
        stability_results[name]["parameter_distance"] = param_distances
        print(f"  parameter distance: mean={np.mean(param_distances):.4f}, std={np.std(param_distances):.4f}")

if not stability_results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Plot — importance-distance distribution per model

In [ ]:
if stability_results:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot(
        [r["importance_distance"] for r in stability_results.values()],
        tick_labels=list(stability_results.keys()),
    )
    ax.set_ylabel("feature-importance distance (bootstrap)")
    plt.tight_layout()
    plt.show()

## Smoke test — remove once real models are in `models/`

Same throwaway logistic regression as the interpretability notebook, on a small
sample and few bootstrap iterations, purely to check the harness runs.

In [ ]:
from sklearn.linear_model import LogisticRegression

class _SmokeTestModule:
    _medians = None  # fixed at fit time so a later all-NaN batch (e.g. a masked coalition) still fills

    @staticmethod
    def fit(X, y):
        Xn = X.select_dtypes("number")
        _SmokeTestModule._medians = Xn.median().fillna(0)
        return LogisticRegression(max_iter=200).fit(Xn.fillna(_SmokeTestModule._medians), y)

    @staticmethod
    def predict_proba(model, X):
        Xn = X.select_dtypes("number").fillna(_SmokeTestModule._medians)
        return model.predict_proba(Xn)

if not stability_results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = get_X_y(sample)
    test_sample = test_df.sample(2_000, random_state=42)
    Xt, yt = get_X_y(test_sample)
    smoke_distances = stability_bootstrap(_SmokeTestModule, "logreg", Xs, ys, n_boot=5)
    print("Smoke test importance distances:", smoke_distances, "— harness is wired correctly.")